In [21]:
print("Day - 06")

Day - 06


In [22]:
# .apply() vs vectorization (when to use which—performance tradeoff)

In [23]:
import pandas as pd
import numpy as np

# Sample DataFrame
df = pd.DataFrame({
    "A": [1, 2, 3, 4],
    "B": [10, 20, 30, 40]
})

# Using .apply() to sum rows
df['row_sum'] = df.apply(lambda row: row['A'] + row['B'], axis=1)
df

,A,B,row_sum
0,1,10,11
1,2,20,22
2,3,30,33
3,4,40,44


In [24]:
df["vec_row_sum"] = df["A"] + df["B"]
df

,A,B,row_sum,vec_row_sum
0,1,10,11,11
1,2,20,22,22
2,3,30,33,33
3,4,40,44,44


In [25]:
df["Mean"] = (df["A"] + df["B"])/2

In [26]:
df

,A,B,row_sum,vec_row_sum,Mean
0,1,10,11,11,5.5
1,2,20,22,22,11.0
2,3,30,33,33,16.5
3,4,40,44,44,22.0


In [27]:
import time

In [11]:
# Big DataFrame
df_big = pd.DataFrame(np.random.randint(0, 100, size=(1000000, 2)), columns=['A', 'B'])
df_big

,A,B
0,38,58
1,77,50
2,66,67
3,9,47
4,38,2
...,...,...
999995,80,86
999996,23,63
999997,30,31
999998,21,56


In [12]:
# .apply()
start = time.time()
df_big['sum_apply'] = df_big.apply(lambda row: row['A'] + row['B'], axis=1)
print("Apply time:", time.time() - start)

Apply time: 19.70048999786377


In [14]:
# Vectorized
start = time.time()
df_big['sum_vec'] = df_big['A'] + df_big['B']
print("Vectorized time:", time.time() - start)

Vectorized time: 0.10421514511108398


In [17]:
# Datetime operations (parsing, resampling for time series)

In [18]:
# 1. Parsing Dates

# When you load a CSV, dates might come as strings. You need to convert them to datetime objects to do operations.

In [42]:
# Sample data
data = {
    "date": ["2026-01-01", "2026-01-02", "2026-01-02", "2026-01-03"],
    "value": [10, 15, 20, 15]
}

df_date = pd.DataFrame(data)
print(df_date.dtypes)


date     object
value     int64
dtype: object


In [43]:
df_date

,date,value
0,2026-01-01,10
1,2026-01-02,15
2,2026-01-02,20
3,2026-01-03,15


In [44]:
df_date['date'] = pd.to_datetime(df_date['date'])
print(df_date.dtypes)

date     datetime64[ns]
value             int64
dtype: object


In [45]:
df_date["year"] = df_date["date"].dt.year
df_date['month'] = df_date['date'].dt.month
df_date['day'] = df_date['date'].dt.day
df_date['weekday'] = df_date['date'].dt.day_name()
df_date

,date,value,year,month,day,weekday
0,2026-01-01,10,2026,1,1,Thursday
1,2026-01-02,15,2026,1,2,Friday
2,2026-01-02,20,2026,1,2,Friday
3,2026-01-03,15,2026,1,3,Saturday


In [46]:
df_date.set_index('date', inplace=True)
df_date

,value,year,month,day,weekday
date,,,,,
2026-01-01,10,2026,1,1,Thursday
2026-01-02,15,2026,1,2,Friday
2026-01-02,20,2026,1,2,Friday
2026-01-03,15,2026,1,3,Saturday


In [50]:
weekly = df_date[["value"]].resample('W').sum()
weekly

,value
date,
2026-01-04,60


In [55]:
df_date['prev_day'] = df_date['value'].shift(1)
df_date


,value,year,month,day,weekday,prev_day
date,,,,,,
2026-01-01,10,2026,1,1,Thursday,NaN
2026-01-02,15,2026,1,2,Friday,10.0
2026-01-02,20,2026,1,2,Friday,15.0
2026-01-03,15,2026,1,3,Saturday,20.0


In [58]:
df_date['7d_ma'] = df_date['value'].rolling(window=2).mean()
df_date

,value,year,month,day,weekday,prev_day,7d_ma
date,,,,,,,
2026-01-01,10,2026,1,1,Thursday,NaN,NaN
2026-01-02,15,2026,1,2,Friday,10.0,12.5
2026-01-02,20,2026,1,2,Friday,15.0,17.5
2026-01-03,15,2026,1,3,Saturday,20.0,17.5


In [59]:
# Parsing Custom Date Formats
# Sometimes your CSV has weird date formats:

In [66]:
df = pd.DataFrame({
    "date": ["01/01/2026", "02/01/2026", "03/01/2026"],
    "value": [10, 15, 20]
})

# Specify format
df['date'] = pd.to_datetime(df['date'], format="%d/%m/%Y")


In [67]:
df

,date,value
0,2026-01-01,10
1,2026-01-02,15
2,2026-01-03,20


In [68]:
# Why memory optimization matters

# Imagine you have a 10 million row CSV with multiple columns.

# By default, pandas uses int64, float64, or object types for data.

# These are big in memory.

# Your computer can slow down or crash when working with large datasets.

# Goal: Reduce memory usage without losing data accuracy.

In [69]:
df = pd.DataFrame({
    "int_col": np.random.randint(0, 100, size=1000000),
    "float_col": np.random.rand(1000000),
    "cat_col": np.random.choice(['A', 'B', 'C'], size=1000000)
})

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   int_col    1000000 non-null  int64  
 1   float_col  1000000 non-null  float64
 2   cat_col    1000000 non-null  object 
dtypes: float64(1), int64(1), object(1)
memory usage: 22.9+ MB
None


In [70]:
df['int_col'] = pd.to_numeric(df['int_col'], downcast='unsigned')  # smaller integer
df['float_col'] = pd.to_numeric(df['float_col'], downcast='float')  # smaller float


In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   int_col    1000000 non-null  uint8  
 1   float_col  1000000 non-null  float32
 2   cat_col    1000000 non-null  object 
dtypes: float32(1), object(1), uint8(1)
memory usage: 12.4+ MB


In [72]:
# Reduces memory drastically for repeating strings (like categories).
df['cat_col'] = df['cat_col'].astype('category')

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype   
---  ------     --------------    -----   
 0   int_col    1000000 non-null  uint8   
 1   float_col  1000000 non-null  float32 
 2   cat_col    1000000 non-null  category
dtypes: category(1), float32(1), uint8(1)
memory usage: 5.7 MB
